In [2]:
# ==============================================================================
# GEREKLİ KÜTÜPHANELERİN YÜKLENMESİ (IMPORTS) VE GPU/CPU AYARI
# ==============================================================================

import torch 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter

# GPU (CUDA) / CPU Cihaz Seçimi (Device Configuration)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")
if torch.cuda.is_available():
    print(f"GPU Modeli: {torch.cuda.get_device_name(0)}")


Kullanılan Cihaz: cuda
GPU Modeli: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
# ==============================================================================
# METİN TEMİZLEME VE PARÇALAMA (TOKENIZATION)
# ==============================================================================

def tokenize(text):
    """
    Ham metni alır; HTML etiketlerini ve noktalama işaretlerini temizler,
    tüm harfleri küçük harfe çevirerek kelimelerine ayırır (token listesi oluşturur).
    """
    # 1. HTML Etiketlerini Temizleme:
    # IMDB yorumlarında sıkça karşılaşılan '<br />' gibi HTML etiketlerini boşlukla değiştiririz.
    text = re.sub(r"<.*?>", "", text)
    
    # 2. Harf Küçültme ve Noktalama İşaretlerini Temizleme:
    # 'text.lower()' ile büyük/küçük harf farkını ortadan kaldırırız ('Movie' ile 'movie' aynı kelime olsun).
    # '[^\w\s]' ifadesi harf, rakam ve boşluk dışındaki tüm karakterleri (noktalama vb.) siler.
    text = re.sub(r"[^\w\s]", "", text.lower())
    
    # 3. Kelimeleri Boşluklara Göre Ayırma:
    # Temizlenen metni boşluklardan bölerek ['this', 'movie', 'is', 'great'] şeklinde bir kelime listesi döndürür.
    return text.split()

In [4]:
# ==============================================================================
# KELİME DAĞARCIĞI OLUŞTURMA (BUILD VOCABULARY)
# ==============================================================================

def build_vocab(texts, max_words=10000):
    """
    Verilen metinlerde en sık geçen kelimeleri bularak her kelimeye benzersiz bir tam sayı (ID) atar.
    Yapay sinir ağları doğrudan kelimelerle değil, bu sayısal ID'ler ile çalışır.
    """
    counter = Counter()
    
    # 1. Tüm metinleri tokenize ederek kelimelerin geçiş sayılarını sayıyoruz
    for text in texts:
        counter.update(tokenize(text))
        
    # 2. En sık geçen kelimeleri seçiyoruz:
    # Toplam kelime kapasitemiz max_words (10000). 
    # 2 tanesini özel etiketler (<PAD> ve <UNK>) için ayırdığımızdan en sık geçen (max_words - 2) kelimeyi alıyoruz.
    most_common = counter.most_common(max_words - 2)
    
    # 3. Özel Token'lar (Special Tokens):
    # <PAD> (ID: 0) -> Cümleleri aynı uzunluğa (max_len=500) getirmek için eklenen dolgu etiketi.
    # <UNK> (ID: 1) -> Kelime dağarcığında bulunmayan (nadir/bilinmeyen) kelimeleri temsil eden etiket.
    vocab = {"<PAD>": 0, "<UNK>": 1}
    
    # 4. En sık geçen kelimeleri sözlüğe sıradaki ID numarası ile ekliyoruz
    for word, _ in most_common:
        vocab[word] = len(vocab)
        
    return vocab

In [5]:
# ==============================================================================
# PYTORCH DATASET SINIFI (IMDBDataset)
# ==============================================================================

class IMDBDataset(Dataset):
    """
    Pandas DataFrame'indeki metin ve etiketleri alıp PyTorch tensörlerine dönüştüren sınıf.
    Her metni kelime ID dizisine çevirir ve sabit uzunlukta (max_len) tutar.
    """
    def __init__(self, df, vocab, max_len=200):
        # 1. Etiketleri Sayısallaştırma: 'positive' -> 1.0, 'negative' -> 0.0
        self.labels = [1.0 if s == "positive" else 0.0 for s in df["sentiment"].values]
        self.max_len = max_len
        self.vocab = vocab
        self.encoded_reviews = []

        # 2. Yorumları Sayısal ID Dizisine Çevirme ve Pre-Padding:
        for text in df["review"].values:
            tokens = tokenize(text)
            ids = [self.vocab.get(token, 1) for token in tokens]
            
            # 3. Pre-padding & Truncation:
            # Başa dolgu (Pre-padding) yapılarak gerçek kelimelerin dizinin sonunda yer alması sağlanır.
            # Böylece RNN son adımlarda dolgu (0) yerine gerçek kelimeleri işler ve hafızası silinmez.
            if len(ids) < max_len:
                ids = [0] * (max_len - len(ids)) + ids
            else:
                ids = ids[:max_len]
                
            self.encoded_reviews.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.encoded_reviews[idx], dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y


In [6]:
# ==============================================================================
# VERİYİ OKUMA VE TRAIN / TEST OLARAK AYIRMA
# ==============================================================================

# 1. IMDB veri setini (CSV formatında) bilgisayardan okuyoruz (50.000 adet film yorumu içerir)
df = pd.read_csv("C:/Users/Emirhan/Desktop/Pytorch ile derin öğrenme/Rnn/data/IMDB Dataset.csv")

# 2. Veri Setini Eğitim (Train) ve Test Olarak İkiye Ayırma:
# Modelin eğitim sırasında görmediği verilerle başarısını ölçmek amacıyla;
# verinin %80'ini eğitim (train_df), kalan %20'sini test (test_df) olarak ayırıyoruz.
# random_state=42: Her çalıştırmada aynı rastgele örneklerin seçilmesini sağlar (tekrarlanabilirlik).
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

# 3. Ayrılan veri sayısını ekrana yazdırıyoruz
print(f"Toplam Veri Sayısı  : {len(df)}")
print(f"Eğitim Seti (Train) : {len(train_df)} adet (%80)")
print(f"Test Seti (Test)    : {len(test_df)} adet (%20)")

Toplam Veri Sayısı  : 50000
Eğitim Seti (Train) : 40000 adet (%80)
Test Seti (Test)    : 10000 adet (%20)


In [7]:
# ==============================================================================
# VOCABULARY OLUŞTURMA VE DATALOADER'LARIN HAZIRLANMASI
# ==============================================================================

# 1. Kelime Dağarcığını (Vocab) SADECE Eğitim Verisi Üzerinden Çıkarma:
vocab = build_vocab(train_df["review"], max_words=10000)

# 2. Eğitim ve Test Dataset Nesnelerinin Oluşturulması (max_len=200):
train_dataset = IMDBDataset(train_df, vocab, max_len=200)
test_dataset = IMDBDataset(test_df, vocab, max_len=200)

# 3. DataLoader Tanımlamaları (batch_size=64):
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 4. İlk batch'i çekerek girdi ve etiket boyutlarını kontrol ediyoruz
sample_x, sample_y = next(iter(train_loader))
print("Girdi Batch Boyutu (Batch Size, Max Len):", sample_x.shape)  # [64, 200]
print("Etiket Batch Boyutu                     :", sample_y.shape)  # [64]


Girdi Batch Boyutu (Batch Size, Max Len): torch.Size([64, 200])
Etiket Batch Boyutu                     : torch.Size([64])


In [8]:
# ==============================================================================
# RNN (RECURRENT NEURAL NETWORK) MODEL MİMARİSİ
# ==============================================================================

class RNN(nn.Module):
    """
    Film yorumlarını analiz edip olumlu ya da olumsuz olarak sınıflandıran RNN mimarisi.
    Kelimelerin bağlamsal gizli durumlarını üretir ve Mean Pooling (Ortalama Havuzlama)
    ile cümlenin genel anlamını özetler. Bu sayede vanilya RNN'lerdeki gradyan kaybolması
    (vanishing gradient) sorununun önüne geçilir.
    """
    def __init__(self, vocab_size=10000, embedding_dim=128, hidden_dim=128, output_dim=1):
        super(RNN, self).__init__()
        
        # 1. Embedding Katmanı:
        # Kelime ID'lerini 128 boyutlu yoğun vektörlere dönüştürür.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 2. RNN Katmanı:
        # input_size=128, hidden_size=128
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        
        # 3. Tam Bağlantılı (Linear) Çıkış Katmanı:
        self.fc1 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x boyutu: [batch_size, sequence_length]
        
        # 1. Kelime ID'lerini vektörlere dönüştürüyoruz: [batch_size, sequence_length, embedding_dim]
        embedded = self.embedding(x)
        
        # 2. Vektörleri RNN katmanından geçiriyoruz:
        # out: [batch_size, sequence_length, hidden_dim]
        out, hidden = self.rnn(embedded)
        
        # 3. Maskeli Mean Pooling (Ortalama Havuzlama):
        # Sadece son gizli duruma bakmak yerine, cümlenin tüm gerçek kelimelerinin
        # gizli durumlarının ortalamasını alıyoruz. Böylece tüm kelimeler sınıflandırmaya
        # doğrudan katkı sağlar ve gradyanlar geriye kolayca akar.
        mask = (x != 0).unsqueeze(-1).float()  # [batch_size, sequence_length, 1]
        sum_hidden = (out * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        pooled = sum_hidden / lengths          # [batch_size, hidden_dim]
        
        # 4. Çıkış skoru (logit): [batch_size, 1]
        output = self.fc1(pooled)
        return output


In [9]:
# ==============================================================================
# MODEL EĞİTİM FONKSİYONU (TRAIN MODEL)
# ==============================================================================

def train_model(model, train_loader, epochs=5, optimizer=None, criterion=None, device=None):
    """
    Modeli belirtilen epoch sayısı boyunca train_loader verileriyle eğitir
    ve her epoch sonunda ortalama kaybı ekrana yazdırır.
    """
    if device is None:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Modeli belirtilen cihaza (GPU/CPU) taşıyoruz
    model.to(device)

    # 1. Modeli eğitim moduna alıyoruz
    model.train()
    run_loss = []
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        # Mini-batch'ler halinde eğitim verisini geziyoruz
        for batch_idx, (x, y) in enumerate(train_loader):
            # Verileri GPU'ya (seçili cihaza) aktarıyoruz
            x, y = x.to(device), y.to(device)

            # Bir önceki adımdan kalan artık gradyanları sıfırlıyoruz
            optimizer.zero_grad()
            
            # İleri yayılım (Forward pass): Modeli çalıştırıp tahmin üretiyoruz
            output = model(x)
            
            # Kayıp (Loss) hesaplama
            loss = criterion(output.squeeze(1), y)
            
            # Geriye yayılım (Backward pass): Gradyanları hesaplıyoruz
            loss.backward()
            
            # RNN'lerde gradyan patlamasını (exploding gradient) önlemek için kırpma uyguluyoruz
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            
            # Optimizasyon: Modelin ağırlık parametrelerini güncelliyoruz
            optimizer.step()
            
            # Batch kaybını topluyoruz
            epoch_loss += loss.item()
            
        # Epoch ortalama kaybını hesaplayıp listeye ekliyoruz
        avg_loss = epoch_loss / len(train_loader)
        run_loss.append(avg_loss)
        
        # Her epoch sonunda güncel loss bilgisini ekrana yazdırıyoruz
        print(f"Epoch {epoch+1:02d}/{epochs:02d} -> Ortalama Kayıp (Train Loss): {avg_loss:.4f}")
        
    return run_loss


In [10]:
# ==============================================================================
# MODEL TEST VE DEĞERLENDİRME FONKSİYONLARI (TEST & INFERENCE)
# ==============================================================================

def evaluate_model(model, test_loader, criterion, device=None):
    """
    Modelin eğitimde hiç görmediği Test Veri Kümesi (test_loader) üzerindeki
    başarısını (Doğruluk / Accuracy) ve ortalama test kaybını hesaplar.
    """
    if device is None:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Modeli belirtilen cihaza taşıyıp değerlendirme (eval) moduna alıyoruz
    model.to(device)
    model.eval()
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    # Test aşamasında gradyan hesaplamasını kapatıyoruz
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)

            output = model(x)
            loss = criterion(output.squeeze(1), y)
            total_loss += loss.item()
            
            probabilities = torch.sigmoid(output.squeeze(1))
            predictions = (probabilities >= 0.5).float()
            correct += (predictions == y).sum().item()
            total += y.size(0)
            
    avg_loss = total_loss / len(test_loader)
    accuracy = (correct / total) * 100
    
    print("-" * 50)
    print(f"TEST VERİ SETİ SONUÇLARI:")
    print(f"Ortalama Test Kaybı (Loss)     : {avg_loss:.4f}")
    print(f"Model Doğruluk Oranı (Accuracy): %{accuracy:.2f} ({correct}/{total} doğru)")
    print("-" * 50)
    
    return avg_loss, accuracy


def predict_sentiment(text, model, vocab, max_len=200, device=None):
    """
    Kullanıcının klavyeden gireceği serbest bir film yorumu metnini alır,
    modelin anlayacağı formata getirir ve tahmin sonucunu (Olumlu/Olumsuz) güven oranıyla döndürür.
    """
    if device is None:
        try:
            device = next(model.parameters()).device
        except StopIteration:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()
    
    # 1. Girilen metni tokenize ediyoruz
    tokens = tokenize(text)
    ids = [vocab.get(token, 1) for token in tokens]
    
    # 2. Pre-padding ile 200 kelimeye sabitliyoruz
    if len(ids) < max_len:
        ids = [0] * (max_len - len(ids)) + ids
    else:
        ids = ids[:max_len]
        
    input_tensor = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        prob = torch.sigmoid(output.squeeze(1)).item()
        
    is_positive = prob >= 0.5
    sentiment = "Olumlu (Positive)" if is_positive else "Olumsuz (Negative)"
    confidence = prob if is_positive else (1.0 - prob)
    
    return sentiment, confidence, prob


In [13]:
if __name__ == "__main__":
    model = RNN(vocab_size=len(vocab), embedding_dim=128, hidden_dim=128, output_dim=1).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = train_model(model, train_loader, epochs=10, optimizer=optimizer, criterion=criterion, device=device)

    evaluate_model(model, test_loader, criterion, device=device)

    sample_reviews = [
        "This movie was an absolute masterpiece! Brilliant acting and great story.",
        "Terrible film, completely boring and a total waste of time."
    ]
    for rev in sample_reviews:
        sentiment, conf, _ = predict_sentiment(rev, model, vocab, device=device)
        print(f"\nYorum: '{rev}'\nTahmin: {sentiment} (%{conf*100:.2f} güven)")


Epoch 01/10 -> Ortalama Kayıp (Train Loss): 0.4767
Epoch 02/10 -> Ortalama Kayıp (Train Loss): 0.3185
Epoch 03/10 -> Ortalama Kayıp (Train Loss): 0.2639
Epoch 04/10 -> Ortalama Kayıp (Train Loss): 0.2289
Epoch 05/10 -> Ortalama Kayıp (Train Loss): 0.2021
Epoch 06/10 -> Ortalama Kayıp (Train Loss): 0.1735
Epoch 07/10 -> Ortalama Kayıp (Train Loss): 0.1507
Epoch 08/10 -> Ortalama Kayıp (Train Loss): 0.1344
Epoch 09/10 -> Ortalama Kayıp (Train Loss): 0.1151
Epoch 10/10 -> Ortalama Kayıp (Train Loss): 0.0944
--------------------------------------------------
TEST VERİ SETİ SONUÇLARI:
Ortalama Test Kaybı (Loss)     : 0.4333
Model Doğruluk Oranı (Accuracy): %85.21 (8521/10000 doğru)
--------------------------------------------------

Yorum: 'This movie was an absolute masterpiece! Brilliant acting and great story.'
Tahmin: Olumlu (Positive) (%100.00 güven)

Yorum: 'Terrible film, completely boring and a total waste of time.'
Tahmin: Olumsuz (Negative) (%100.00 güven)
